# CBP 2018–2023 Cross-Year EDA & Validation

**DATA 690 Census Bureau Capstone — Win**

## Purpose

This notebook is the third stage of the County Business Patterns (CBP) workflow.

It does **not** retrieve new API data. Instead, it uses the two previously validated candidate packages:

- `cbp_candidate_outputs_2018.zip`
- `cbp_candidate_outputs_2023.zip`

The notebook will:

1. Upload and extract both validated candidate packages.
2. Read the CSV and provenance metadata.
3. Verify cross-year schema, geography, industry classification, units, and validation status.
4. Preserve classification codes as strings.
5. Compare disclosure/noise information.
6. Create a matched 2018–2023 analytical dataset.
7. Calculate absolute and percentage changes for establishments, employment, Q1 payroll, and annual payroll.
8. Produce exploratory tables and charts.
9. Evaluate candidate measures for Bar Chart, Slope Diagram, and Table examples.
10. Save a reusable cross-year dataset, validation report, and EDA outputs.

> **Important:** The notebook separates automated comparability checks from analytical interpretation. A PASS means the tested structures are compatible; it does not by itself establish causation, statistical significance, or inflation-adjusted economic growth.

## 1. Upload the Two Validated Candidate ZIP Files

When prompted, upload both ZIP files from the earlier tests.

In [ ]:
from google.colab import files
from pathlib import Path
import io
import json
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

uploaded = files.upload()

expected = {
    "cbp_candidate_outputs_2018.zip",
    "cbp_candidate_outputs_2023.zip",
}

missing = expected - set(uploaded)
if missing:
    raise FileNotFoundError(
        f"Missing required file(s): {sorted(missing)}. "
        "Please rerun this cell and upload both validated ZIP packages."
    )

WORK = Path("cbp_crossyear_work")
WORK.mkdir(exist_ok=True)

for filename in sorted(expected):
    target = WORK / filename.replace(".zip", "")
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[filename]), "r") as z:
        z.extractall(target)

print("Both packages extracted successfully.")

## 2. Load Candidate Data and Metadata

Classification codes are explicitly loaded as strings so values such as `00` and `001` retain their leading zeros.

In [ ]:
def load_package(year):
    folder = WORK / f"cbp_candidate_outputs_{year}"
    csv_path = folder / f"cbp_{year}_candidate.csv"
    meta_path = folder / f"cbp_{year}_metadata.json"

    if not csv_path.exists() or not meta_path.exists():
        raise FileNotFoundError(
            f"Expected files were not found for {year}: {csv_path}, {meta_path}"
        )

    dtype_map = {
        "state": "string",
        "NAICS2017": "string",
        "NAICS2022": "string",
        "LFO": "string",
        "EMPSZES": "string",
        "YEAR": "string",
        "naics_code_requested": "string",
    }

    df = pd.read_csv(csv_path, dtype=dtype_map)

    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    return df, meta

df18, meta18 = load_package(2018)
df23, meta23 = load_package(2023)

print("2018 shape:", df18.shape)
print("2023 shape:", df23.shape)
print("2018 validation:", meta18.get("validation_status"))
print("2023 validation:", meta23.get("validation_status"))

display(df18.head())
display(df23.head())

## 3. Inspect Provenance Side by Side

This step makes the comparison configuration visible before any analytical calculations.

In [ ]:
provenance = pd.DataFrame([
    {
        "Year": 2018,
        "Validation": meta18.get("validation_status"),
        "Geography": meta18.get("geography"),
        "NAICS variable": meta18.get("naics_variable"),
        "NAICS code": meta18.get("naics_code"),
        "LFO": meta18.get("legal_form_code"),
        "EMPSZES": meta18.get("employment_size_code"),
        "Rows": meta18.get("candidate_rows"),
        "Endpoint": meta18.get("api_endpoint"),
    },
    {
        "Year": 2023,
        "Validation": meta23.get("validation_status"),
        "Geography": meta23.get("geography"),
        "NAICS variable": meta23.get("naics_variable"),
        "NAICS code": meta23.get("naics_code"),
        "LFO": meta23.get("legal_form_code"),
        "EMPSZES": meta23.get("employment_size_code"),
        "Rows": meta23.get("candidate_rows"),
        "Endpoint": meta23.get("api_endpoint"),
    }
])

display(provenance)

## 4. Cross-Year Comparability Validation

The following tests must be reviewed before calculating change over time.

A distinction is made between:

- **PASS** — tested condition is compatible.
- **WARNING** — review is required before interpretation.
- **FAIL** — do not proceed with the affected comparison.

In [ ]:
checks = []

def add_check(name, status, detail):
    checks.append({"Check": name, "Status": status, "Detail": detail})

# Prior validation
both_valid = (
    meta18.get("validation_status") == "PASS"
    and meta23.get("validation_status") == "PASS"
)
add_check(
    "Source candidate validation",
    "PASS" if both_valid else "FAIL",
    f"2018={meta18.get('validation_status')}; 2023={meta23.get('validation_status')}"
)

# Geography type
same_geo = meta18.get("geography") == meta23.get("geography")
add_check(
    "Geography level",
    "PASS" if same_geo else "FAIL",
    f"2018={meta18.get('geography')}; 2023={meta23.get('geography')}"
)

# State coverage
states18 = set(df18["state"].astype(str).str.zfill(2))
states23 = set(df23["state"].astype(str).str.zfill(2))
same_states = states18 == states23
add_check(
    "Geography coverage",
    "PASS" if same_states else "FAIL",
    f"2018 states={sorted(states18)}; 2023 states={sorted(states23)}"
)

# Core measures
core = ["ESTAB", "EMP", "PAYQTR1", "PAYANN"]
missing18 = [c for c in core if c not in df18.columns]
missing23 = [c for c in core if c not in df23.columns]
add_check(
    "Core analytical measures",
    "PASS" if not missing18 and not missing23 else "FAIL",
    f"2018 missing={missing18 or 'None'}; 2023 missing={missing23 or 'None'}"
)

# Industry concept/code
code18 = str(meta18.get("naics_code", ""))
code23 = str(meta23.get("naics_code", ""))
same_code = code18 == code23
add_check(
    "Requested NAICS code",
    "PASS" if same_code else "WARNING",
    f"2018={code18}; 2023={code23}"
)

# NAICS vintage
vintage18 = meta18.get("naics_variable")
vintage23 = meta23.get("naics_variable")
same_vintage = vintage18 == vintage23
add_check(
    "NAICS vintage",
    "PASS" if same_vintage else "WARNING",
    f"2018={vintage18}; 2023={vintage23}. "
    "If vintages differ, confirm the selected aggregate industry concept remains comparable."
)

# LFO
same_lfo = str(meta18.get("legal_form_code")) == str(meta23.get("legal_form_code"))
add_check(
    "Legal-form slice",
    "PASS" if same_lfo else "FAIL",
    f"2018={meta18.get('legal_form_code')}; 2023={meta23.get('legal_form_code')}"
)

# Employment size
same_size = str(meta18.get("employment_size_code")) == str(meta23.get("employment_size_code"))
add_check(
    "Employment-size slice",
    "PASS" if same_size else "FAIL",
    f"2018={meta18.get('employment_size_code')}; 2023={meta23.get('employment_size_code')}"
)

# Noise/disclosure columns
noise = ["EMP_N", "PAYQTR1_N", "PAYANN_N"]
noise18 = [c for c in noise if c in df18.columns]
noise23 = [c for c in noise if c in df23.columns]
add_check(
    "Noise/disclosure fields",
    "PASS" if set(noise18) == set(noise23) and len(noise18) == len(noise) else "WARNING",
    f"2018={noise18}; 2023={noise23}"
)

validation = pd.DataFrame(checks)
display(validation)

if (validation["Status"] == "FAIL").any():
    CROSSYEAR_STATUS = "FAIL"
elif (validation["Status"] == "WARNING").any():
    CROSSYEAR_STATUS = "WARNING"
else:
    CROSSYEAR_STATUS = "PASS"

print("CROSS-YEAR STRUCTURAL STATUS:", CROSSYEAR_STATUS)

## 5. Review Industry Labels

Even when the requested code is the same, inspect the returned labels before treating the industry concept as equivalent.

In [ ]:
def naics_label_column(df):
    matches = [c for c in df.columns if c.startswith("NAICS") and c.endswith("_LABEL")]
    return matches[0] if matches else None

label18_col = naics_label_column(df18)
label23_col = naics_label_column(df23)

labels = pd.DataFrame({
    "Year": [2018, 2023],
    "NAICS variable": [vintage18, vintage23],
    "Requested code": [code18, code23],
    "Returned label": [
        df18[label18_col].dropna().iloc[0] if label18_col and not df18[label18_col].dropna().empty else None,
        df23[label23_col].dropna().iloc[0] if label23_col and not df23[label23_col].dropna().empty else None,
    ]
})
display(labels)

## 6. Review Disclosure / Noise Information

Do not interpret noise-range codes as ordinary analytical measures.  
This section simply confirms that the information has been retained and shows the observed values by year.

In [ ]:
noise_summary_rows = []

for year, df in [(2018, df18), (2023, df23)]:
    for col in noise:
        if col in df.columns:
            vals = sorted(df[col].dropna().astype(str).unique().tolist())
            noise_summary_rows.append({
                "Year": year,
                "Field": col,
                "Nonmissing": int(df[col].notna().sum()),
                "Observed values": ", ".join(vals)
            })

noise_summary = pd.DataFrame(noise_summary_rows)
display(noise_summary)

## 7. Build the Matched 2018–2023 Analytical Dataset

Rows are matched by state FIPS. Core measures are converted to numeric before change calculations.

In [ ]:
def prepare(df, year):
    out = df.copy()
    out["state"] = out["state"].astype(str).str.zfill(2)

    for c in core:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    keep = ["state", "NAME"] + core + [c for c in noise if c in out.columns]
    out = out[keep].copy()

    rename = {c: f"{c}_{year}" for c in out.columns if c not in ["state", "NAME"]}
    rename["NAME"] = f"NAME_{year}"
    return out.rename(columns=rename)

a18 = prepare(df18, 2018)
a23 = prepare(df23, 2023)

merged = a18.merge(a23, on="state", how="outer", validate="one_to_one")

merged["Geography"] = merged["NAME_2023"].fillna(merged["NAME_2018"])

for measure in core:
    merged[f"{measure}_change"] = merged[f"{measure}_2023"] - merged[f"{measure}_2018"]
    merged[f"{measure}_pct_change"] = np.where(
        merged[f"{measure}_2018"] != 0,
        merged[f"{measure}_change"] / merged[f"{measure}_2018"] * 100,
        np.nan
    )

cols = (
    ["state", "Geography"]
    + [f"{m}_2018" for m in core]
    + [f"{m}_2023" for m in core]
    + [f"{m}_change" for m in core]
    + [f"{m}_pct_change" for m in core]
)

analysis = merged[cols].copy()
display(analysis)

## 8. Human-Readable Change Summary

Payroll fields are reported by the API in thousands of dollars.  
For readability, this exploratory table converts annual payroll to billions of dollars.

**Important:** 2018 and 2023 payroll values are nominal values unless a separate inflation adjustment is introduced.

In [ ]:
summary = pd.DataFrame({
    "Geography": analysis["Geography"],
    "Establishments 2018": analysis["ESTAB_2018"],
    "Establishments 2023": analysis["ESTAB_2023"],
    "Establishments change %": analysis["ESTAB_pct_change"],
    "Employment 2018": analysis["EMP_2018"],
    "Employment 2023": analysis["EMP_2023"],
    "Employment change %": analysis["EMP_pct_change"],
    "Annual payroll 2018 ($B nominal)": analysis["PAYANN_2018"] / 1_000_000,
    "Annual payroll 2023 ($B nominal)": analysis["PAYANN_2023"] / 1_000_000,
    "Annual payroll change % (nominal)": analysis["PAYANN_pct_change"],
})

display(
    summary.style.format({
        "Establishments 2018": "{:,.0f}",
        "Establishments 2023": "{:,.0f}",
        "Establishments change %": "{:.1f}%",
        "Employment 2018": "{:,.0f}",
        "Employment 2023": "{:,.0f}",
        "Employment change %": "{:.1f}%",
        "Annual payroll 2018 ($B nominal)": "${:,.1f}",
        "Annual payroll 2023 ($B nominal)": "${:,.1f}",
        "Annual payroll change % (nominal)": "{:.1f}%",
    })
)

## 9. Exploratory Chart — Employment, 2023

This is an EDA chart used to assess whether employment is a useful candidate for a later Bar Chart example.

In [ ]:
plot_df = analysis.sort_values("EMP_2023")

plt.figure(figsize=(9, 5))
plt.barh(plot_df["Geography"], plot_df["EMP_2023"])
plt.xlabel("Employment")
plt.ylabel("")
plt.title("CBP Employment by Selected Geography, 2023")
plt.ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()

## 10. Exploratory Chart — Establishment Change, 2018–2023

This exploratory view evaluates whether percentage change in establishments produces a useful comparison story.

In [ ]:
plot_df = analysis.sort_values("ESTAB_pct_change")

plt.figure(figsize=(9, 5))
plt.barh(plot_df["Geography"], plot_df["ESTAB_pct_change"])
plt.axvline(0, linewidth=1)
plt.xlabel("Percent change")
plt.ylabel("")
plt.title("Change in CBP Establishments, 2018–2023")
plt.tight_layout()
plt.show()

## 11. Exploratory Slope View — Annual Payroll

This chart is **exploratory only**.

Because payroll is monetary, the values are labeled as **nominal dollars**.  
Do not describe the difference as real economic growth unless an appropriate inflation adjustment is added and documented.

In [ ]:
slope = analysis[["Geography", "PAYANN_2018", "PAYANN_2023"]].copy()
slope["2018_B"] = slope["PAYANN_2018"] / 1_000_000
slope["2023_B"] = slope["PAYANN_2023"] / 1_000_000

plt.figure(figsize=(10, 6))

for _, row in slope.iterrows():
    plt.plot([2018, 2023], [row["2018_B"], row["2023_B"]], marker="o")
    plt.text(2017.9, row["2018_B"], f'{row["Geography"]}  ${row["2018_B"]:.1f}B',
             ha="right", va="center", fontsize=8)
    plt.text(2023.1, row["2023_B"], f'${row["2023_B"]:.1f}B',
             ha="left", va="center", fontsize=8)

plt.xticks([2018, 2023])
plt.ylabel("Annual payroll ($ billions, nominal)")
plt.title("Exploratory CBP Annual Payroll Comparison, 2018–2023")
plt.xlim(2016.6, 2024.4)
plt.tight_layout()
plt.show()

## 12. Candidate Visualization Assessment

The notebook evaluates three primary implementation candidates:

### Bar Chart
**Recommended candidate measure:** 2023 employment  
Reason: directly comparable count measure across selected geographies and easier to interpret than a cross-year monetary measure.

### Slope Diagram
**Candidate measure:** 2018 vs. 2023 annual payroll  
Condition: clearly label values as nominal dollars, or perform and document an inflation adjustment before making real-dollar interpretations.

### Table
**Recommended:** establishments, employment, Q1 payroll, and annual payroll for the selected geographies, with year and units explicitly documented.

These are development recommendations, not automatic statistical conclusions.

In [ ]:
assessment = pd.DataFrame([
    {
        "Visualization": "Bar Chart",
        "Candidate measure": "2023 employment",
        "Strength": "Direct state comparison; non-monetary count",
        "Key caution": "Document CBP scope and disclosure/noise treatment",
        "Development recommendation": "Strong candidate"
    },
    {
        "Visualization": "Slope Diagram",
        "Candidate measure": "2018 vs 2023 annual payroll",
        "Strength": "Clear before/after comparison",
        "Key caution": "Nominal dollars; inflation affects interpretation",
        "Development recommendation": "Use with explicit nominal-dollar note or inflation adjustment"
    },
    {
        "Visualization": "Table",
        "Candidate measure": "ESTAB, EMP, PAYQTR1, PAYANN",
        "Strength": "Supports exact-value lookup and reproducibility",
        "Key caution": "State units and disclosure/noise context",
        "Development recommendation": "Strong candidate"
    },
])
display(assessment)

## 13. Save Cross-Year Outputs

The analytical dataset and validation evidence are saved separately from the original candidate snapshots.

In [ ]:
OUT = Path("cbp_crossyear_outputs")
OUT.mkdir(exist_ok=True)

analysis_path = OUT / "cbp_2018_2023_crossyear_analysis.csv"
validation_path = OUT / "cbp_2018_2023_validation.csv"
summary_path = OUT / "cbp_2018_2023_summary.csv"
assessment_path = OUT / "cbp_visualization_candidate_assessment.csv"

analysis.to_csv(analysis_path, index=False)
validation.to_csv(validation_path, index=False)
summary.to_csv(summary_path, index=False)
assessment.to_csv(assessment_path, index=False)

report = {
    "project": "DATA 690 Census Bureau Capstone",
    "dataset": "County Business Patterns (CBP)",
    "comparison_years": [2018, 2023],
    "crossyear_structural_status": CROSSYEAR_STATUS,
    "source_validation": {
        "2018": meta18.get("validation_status"),
        "2023": meta23.get("validation_status")
    },
    "naics": {
        "2018_variable": vintage18,
        "2023_variable": vintage23,
        "2018_code": code18,
        "2023_code": code23
    },
    "geography_fips": sorted(states18 & states23),
    "core_measures": core,
    "important_interpretation_note": (
        "Payroll comparisons are nominal unless an explicit inflation adjustment is performed."
    ),
    "validation_checks": checks
}

report_path = OUT / "cbp_2018_2023_crossyear_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Saved:")
for p in [analysis_path, validation_path, summary_path, assessment_path, report_path]:
    print(" -", p)

## 14. Package the Cross-Year Results

Download this ZIP and retain it as evidence of the EDA/validation stage.

In [ ]:
import zipfile

zip_path = Path("cbp_2018_2023_crossyear_outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file():
            z.write(p, arcname=p.name)

print(f"Created: {zip_path}")
print("Package contents:")
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(" -", name)

files.download(str(zip_path))

## 15. Decision Gate Before Publication-Quality Visualization

Proceed to final visualization development only after reviewing:

- source candidate validation results;
- cross-year structural validation;
- NAICS vintage and industry-label compatibility;
- geography and unit consistency;
- disclosure/noise information;
- the analytical question each chart will answer;
- whether payroll will remain nominal or be inflation-adjusted.

### Workflow status

**CBP API → 2018/2023 candidates → individual validation → cross-year validation & EDA → chart selection → Python/R implementation → cross-language validation → accessible outputs → guidance-page integration**